# Imbalanced Data in Machine Learning

This notebook is an end-to-end, practical guide covering class imbalance and the important techniques used to handle it in Machine Learning.

# 1. What is Class Imbalance?

### Concept
An **imbalanced dataset** is one where the classes are not represented equally. 
* **Majority class:** The class with a significantly higher number of instances.
* **Minority class:** The class with a significantly lower number of instances.
* **Balanced dataset:** Classes have roughly equal frequencies (e.g., 50% vs 50%).
* **Imbalanced dataset:** One class dominates heavily over the other (e.g., 95% vs 5%).

> Key Idea: A simple example of class imbalance is a dataset where 95% of the data belongs to Class 0, and only 5% belongs to Class 1.

**Why this creates problems:**
Machine learning algorithms are typically designed to maximize accuracy and reduce error. When trained on imbalanced data, the model might simply learn to predict the majority class for everything. It achieves high accuracy but fails entirely to identify the minority class, which is often the class we actually care about.\n

# 2. Real-World Examples

### Practical Examples
Class imbalance is extremely common in real-world scenarios:
* **Fraud detection:** Out of millions of credit card transactions, only a tiny fraction are fraudulent.
* **Disease detection:** In a medical screening test, most patients are healthy, while very few have the rare disease.
* **Spam detection:** 90% of emails might be normal, while only 10% are spam.
* **Manufacturing defect detection:** In a factory line producing thousands of items, only a few might be defective.
* **Intrusion detection:** Most network traffic is legitimate, while cyberattacks constitute a small minority of requests.

> Important: In all these cases, the **minority class is often the most important class**. A model that fails to detect fraud or a life-threatening disease is useless, even if it is 99% accurate on normal cases.\n

# 3. Why Accuracy Can Be Misleading

### Concept
Imagine a medical dataset with 1000 samples:
* **950 healthy patients (Class 0)**
* **50 sick patients (Class 1)**

If our model predicts "healthy (Class 0)" for every single patient, let's see what happens to the accuracy.\n

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

# Create synthetic true labels and predictions
y_true = np.array([0] * 950 + [1] * 50)
y_pred = np.array([0] * 1000) # Model always predicts 0

acc = accuracy_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print(f"Accuracy: {acc * 100:.2f}%")
print(f"Minority-class Recall: {recall * 100:.2f}%")\n

> Common Mistake: Evaluating imbalanced datasets using only Accuracy.

Although the accuracy is 95%, the model caught **0%** of the sick patients. Because the model completely failed at its real purpose (identifying sick patients), it is actually a very poor model. Accuracy hides this failure.\n

# 4. Confusion Matrix for Imbalanced Data

### Concept
A Confusion Matrix helps visualize the types of errors a model makes:
* **True Positive (TP):** Sick patient correctly identified as sick.
* **True Negative (TN):** Healthy patient correctly identified as healthy.
* **False Positive (FP):** Healthy patient incorrectly identified as sick (False Alarm).
* **False Negative (FN):** Sick patient incorrectly identified as healthy (Missed Case - typically very dangerous).

For imbalanced data, analyzing the True Positives and False Negatives of the minority class is critical.\n

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Healthy (0)", "Sick (1)"])

plt.figure(figsize=(6,4))
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title("Confusion Matrix: Always Predict 0")
plt.show()\n

Notice how TP is 0 and FN is 50. All minority class cases were predicted wrong (missed!).\n

# 5. Important Metrics for Imbalanced Data

### Concept
Because Accuracy is misleading, we use alternative metrics:

* **Precision:** Out of all cases the model predicted as positive, how many were actually positive? $$\text{Precision} = \frac{TP}{TP + FP}$$
* **Recall (Sensitivity):** Out of all actual positive cases, how many did the model correctly identify? $$\text{Recall} = \frac{TP}{TP + FN}$$
* **Specificty:** Out of all actual negative cases, how many did the model identify as negative? $$\text{Specificity} = \frac{TN}{TN + FP}$$
* **F1 Score:** The harmonic mean of Precision and Recall. Useful when you need to balance both. $$\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$
* **ROC-AUC (Area Under the Receiver Operating Characteristic Curve):** Measures the ability of the model to distinguish between classes.
* **Precision-Recall AUC (Average Precision):** Area under the Precision-Recall curve. Especially useful for heavily imbalanced data.

| Metric | Focus | When to use? |
| --- | --- | --- |
| **Accuracy** | Overall correctness | Balanced data |
| **Precision** | Missing False Positives | When false alarms are costly (e.g., spam filter blocking real work emails) |
| **Recall** | Missing False Negatives | When missing a positive is deadly (e.g., disease detection) |
| **F1 Score** | Balancing Precision/Recall | When both FP and FN are costly |
| **ROC-AUC** | Class separability | Measuring overall ranking power across thresholds |
| **PR-AUC** | Minority class ranking | Highly imbalanced problems evaluating positive class |\n

# 6. Precision vs Recall for Imbalanced Problems

### Concept
There is usually a trade-off between Precision and Recall.

* **High Recall:** The model casts a wide net. It catches almost all minority-class cases, but might also generate many false alarms (low precision).
* **High Precision:** The model is very conservative. When it predicts the minority class, it is usually correct, but it might miss many real cases (low recall).

**Examples:**
* **Disease screening:** A False Negative (missing a disease) can be fatal. A False Positive just means a follow-up test. **Recall is critical.**
* **Fraud investigation (manual):** If there's a team of 5 people reviewing alerts, sending them 100,000 false alarms will overwhelm them. **Precision is also important here.**

> Remember: The metric choice purely depends on the real-world cost of errors.\n

# 7. Creating an Imbalanced Dataset

Let's use `sklearn.datasets.make_classification` to create a synthetic binary classification dataset to demonstrate these concepts.\n

In [ ]:
from sklearn.datasets import make_classification
from collections import Counter

# Create a dataset with 95% Class 0 and 5% Class 1
X, y = make_classification(
    n_samples=5000, 
    n_features=2, 
    n_redundant=0, 
    n_classes=2, 
    weights=[0.95], # 95% majority class
    random_state=42, 
    n_clusters_per_class=1
)

counts = Counter(y)
perc_0 = (counts[0] / len(y)) * 100
perc_1 = (counts[1] / len(y)) * 100

print(f"Total samples: {len(y)}")
print(f"Class 0 (Majority): {counts[0]} ({perc_0:.1f}%)")
print(f"Class 1 (Minority): {counts[1]} ({perc_1:.1f}%)")

plt.figure(figsize=(8,6))
sns.scatterplot(x=X[:,0], y=X[:,1], hue=y, palette=['blue', 'red'], alpha=0.6)
plt.title("Synthetic Imbalanced Dataset")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend(["Class 0 (Majority)", "Class 1 (Minority)"])
plt.show()\n

# 8. Baseline Model on Imbalanced Data

Let's train a Logistic Regression model as our baseline and see its performance. We will evaluate properly by looking beyond accuracy.\n

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# Stratified split to maintain class distribution in train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# Train baseline Logistic Regression
lr_base = LogisticRegression(random_state=42)
lr_base.fit(X_train, y_train)

# Predict
y_pred_base = lr_base.predict(X_test)
y_prob_base = lr_base.predict_proba(X_test)[:, 1]

# Evaluate
print("=== Baseline Model Metrics ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_base):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_base):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_base):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_base):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_base):.4f}")
print(f"Average Precision (PR-AUC): {average_precision_score(y_test, y_prob_base):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_base))\n

The accuracy is very high (~95%), but the **Recall** and **F1 Score** for Class 1 are usually terrible. This perfectly illustrates why looking only at accuracy gives an incomplete picture.\n

# 9. Stratified Train-Test Split

### Concept
When splitting an imbalanced dataset into training and testing sets, a random split can randomly exclude many minority class samples from the test set, making evaluation highly inaccurate.

> Important: Always use `stratify=y` when calling `train_test_split()` for imbalanced classification. It ensures the train and test sets have the exact same proportion of classes as the original data.\n

In [ ]:
# Without stratification (Not recommended)
X_train_unstrat, X_test_unstrat, y_train_unstrat, y_test_unstrat = train_test_split(X, y, test_size=0.3, random_state=42)

print("Original Data Class 1 Proportion:", np.mean(y == 1))
print("Unstratified Test Class 1 Proportion:", np.mean(y_test_unstrat == 1))
print("Stratified Test Class 1 Proportion:", np.mean(y_test == 1))\n

# 10. Class Weights

### Concept
Instead of balancing the data, we can instruct the algorithm to penalize errors on the minority class more severely.
By setting `class_weight="balanced"`, the model gives more weight (importance) to minority samples relative to their frequency.

Let's test this with Logistic Regression, Decision Trees and Random Forests.\n

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

models_weighted = {
    "Logistic Regression (Balanced)": LogisticRegression(class_weight="balanced", random_state=42),
    "Decision Tree (Balanced)": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest (Balanced)": RandomForestClassifier(class_weight="balanced", random_state=42)
}

for name, model in models_weighted.items():
    model.fit(X_train, y_train)
    y_pred_w = model.predict(X_test)
    y_prob_w = model.predict_proba(X_test)[:, 1]
    
    print(f"--- {name} ---")
    print(f"Precision: {precision_score(y_test, y_pred_w):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred_w):.4f}")
    print(f"F1 Score: {f1_score(y_test, y_pred_w):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_w):.4f}")
    print(f"Avg Precision: {average_precision_score(y_test, y_prob_w):.4f}\n")\n

Notice the trade-off here: Recall for the minority class greatly increases, but Precision typically drops (more false positives).\n

# 11. Undersampling

### Concept
**Undersampling** involves randomly removing samples from the **majority class** until it matches the size of the minority class.

* **Advantages:** Faster training, balances classes perfectly.
* **Disadvantages:** Information loss! Throws away potentially useful data from the majority class.\n

In [ ]:
# Simple manual undersampling using Pandas/Numpy
df_train = pd.DataFrame(X_train, columns=['F1', 'F2'])
df_train['target'] = y_train

majority_class = df_train[df_train['target'] == 0]
minority_class = df_train[df_train['target'] == 1]

print("Before Undersampling:")
print(f"Majority: {len(majority_class)}, Minority: {len(minority_class)}")

# Randomly sample the majority class to match minority size
majority_undersampled = majority_class.sample(n=len(minority_class), random_state=42)

df_train_undersampled = pd.concat([majority_undersampled, minority_class])
y_train_under = df_train_undersampled['target'].values
X_train_under = df_train_undersampled[['F1', 'F2']].values

print("\nAfter Undersampling:")
temp_counts = pd.Series(y_train_under).value_counts()
print(f"Majority: {temp_counts[0]}, Minority: {temp_counts[1]}")\n

# 12. Oversampling

### Concept
**Oversampling** involves randomly duplicating samples from the **minority class** until it matches the size of the majority class.

* **Advantages:** No information loss.
* **Disadvantages:** Copies exact same data points, increasing risk of overfitting.\n

In [ ]:
from imblearn.over_sampling import RandomOverSampler

print("Before Oversampling:")
print(pd.Series(y_train).value_counts())

ros = RandomOverSampler(random_state=42)
X_train_over, y_train_over = ros.fit_resample(X_train, y_train)

print("\nAfter Oversampling:")
print(pd.Series(y_train_over).value_counts())\n

> Important: Only resample the **TRAINING data**, NEVER the test set.\n

# 13. Random Oversampling vs Random Undersampling

### Concept

| Method | Main Idea | Advantage | Disadvantage |
|--------|-----------|-----------|--------------|
| **Random Oversampling** | Duplicate minority class samples | Keeps all original majority data, no info loss | May cause severe overfitting due to exact duplicates |
| **Random Undersampling** | Remove majority class samples | Faster training (less data), less memory footprint | Discards potentially highly useful information from the majority class |\n

# 14. SMOTE

### Concept
**Synthetic Minority Over-sampling Technique (SMOTE)** overcomes the overfitting problem of Random Oversampling.
Instead of simply copying minority point duplicates, SMOTE creates **synthetic** new minority samples by interpolating between existing minority examples.

1. It selects a minority sample.
2. It finds its nearest minority neighbors.
3. It creates an artificial sample on the line segment connecting them.

If `imblearn` is unavailable, you can install it via: `!pip install imbalanced-learn`\n

In [ ]:
from imblearn.over_sampling import SMOTE

# Instantiate SMOTE
smote = SMOTE(random_state=42)\n

# 15. SMOTE Example

Let's test SMOTE on our data and compare it against the baseline.\n

In [ ]:
# 1 & 2. Apply SMOTE ONLY to training data
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 3 & 4. Check distributions
print("Before SMOTE:")
print(pd.Series(y_train).value_counts())
print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())

# 5. Train classifier
lr_smote = LogisticRegression(random_state=42)
lr_smote.fit(X_train_smote, y_train_smote)

# 6. Evaluate on untouched test set
y_pred_smote = lr_smote.predict(X_test)
y_prob_smote = lr_smote.predict_proba(X_test)[:, 1]

print("\n=== Baseline vs SMOTE ===")
print(f"Baseline Recall: {recall_score(y_test, y_pred_base):.4f} | SMOTE Recall: {recall_score(y_test, y_pred_smote):.4f}")
print(f"Baseline Precision: {precision_score(y_test, y_pred_base):.4f} | SMOTE Precision: {precision_score(y_test, y_pred_smote):.4f}")
print(f"Baseline F1: {f1_score(y_test, y_pred_base):.4f} | SMOTE F1: {f1_score(y_test, y_pred_smote):.4f}")
print(f"Baseline AP: {average_precision_score(y_test, y_prob_base):.4f} | SMOTE AP: {average_precision_score(y_test, y_prob_smote):.4f}")\n

# 16. Why SMOTE Must Not Be Applied Before Splitting

### Concept
> Common Mistake: Creating synthetic samples via SMOTE on the full dataset, then doing a train/test split.

**Wrong Flow:**
Dataset -> SMOTE -> Train/Test Split
* **Data Leakage Problem:** SMOTE uses existing points to create new synthetic points. If you apply SMOTE first, some synthetic points in the TEST set will be heavily based on points in the TRAIN set. The model will unfairly "know" information about the test data.

**Correct Flow:**
Dataset -> Train/Test Split -> SMOTE on Training Only -> Train Model -> Test on **Untouched** Test Set.
* This ensures the test set truly resembles real-world unseen data with its natural class imbalance.\n

# 17. Imbalanced-Learn Pipeline

### Concept
To avoid data leakage during Cross-Validation, we MUST use pipelines that apply SMOTE only inside each fold's training portion. `sklearn` pipelines don't natively support resampling, so we use `imblearn.pipeline`. \n

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import cross_validate

pipeline = ImbPipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(random_state=42))
])

# Cross-Validation happens correctly: folds split first, THEN smote on training folds
cv_results = cross_validate(pipeline, X_train, y_train, cv=5, scoring=['recall', 'f1', 'average_precision'])

print("Cross-Validation Mean Recall:", np.mean(cv_results['test_recall']))
print("Cross-Validation Mean F1:", np.mean(cv_results['test_f1']))
print("Cross-Validation Mean AP:", np.mean(cv_results['test_average_precision']))\n

# 18. Precision-Recall Curve

### Concept
A Precision-Recall Curve illustrates the trade-off for different threshold levels. It calculates Precision and Recall at various decision thresholds. It is highly informative for imbalanced classification because it ignores True Negatives (majority class performance).\n

In [ ]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_base)

plt.figure(figsize=(7,5))
plt.plot(recalls, precisions, label=f'Baseline AP = {average_precision_score(y_test, y_prob_base):.2f}', color='blue')
plt.title("Precision-Recall Curve (Baseline)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(loc="lower left")
plt.grid(True)
plt.show()\n

# 19. ROC Curve vs Precision-Recall Curve

### Concept

| Feature | ROC-AUC | PR-AUC |
|---------|---------|--------|
| **Focus** | True Positive Rate (Recall) vs False Positive Rate | Precision vs Recall |
| **Imbalanced Data** | Can falsely show great performance due to massive TNs | Often more informative |
| **Minority Class Focus** | Indirectly evaluates both classes | More directly evaluates finding the positive class |

If you care equally about both classes, use ROC. If you care immensely about the rare minority positive class (e.g., fraud), PR-AUC is usually the better guiding metric.\n

# 20. Threshold Tuning

### Concept
Machine Learning algorithms like Logistic Regression predict a **probability** (e.g. 0.85). We decide the class using a threshold.
* Default threshold = `0.50`
* Lowering threshold (e.g., to `0.20`): Flags more points as positive. (Increases Recall, Decreases Precision)
* Raising threshold (e.g., to `0.80`): Stricter. (Decreases Recall, Increases Precision)\n

In [ ]:
thresholds_to_test = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

results = []
for t in thresholds_to_test:
    # Manual threshold mapping
    pred_t = (y_prob_base >= t).astype(int) 
    
    results.append({
        "Threshold": t,
        "Precision": precision_score(y_test, pred_t, zero_division=0),
        "Recall": recall_score(y_test, pred_t),
        "F1": f1_score(y_test, pred_t)
    })

df_thresholds = pd.DataFrame(results)
print(df_thresholds)

plt.figure(figsize=(8,5))
plt.plot(df_thresholds['Threshold'], df_thresholds['Precision'], label='Precision', marker='o')
plt.plot(df_thresholds['Threshold'], df_thresholds['Recall'], label='Recall', marker='s')
plt.plot(df_thresholds['Threshold'], df_thresholds['F1'], label='F1 Score', marker='^', linestyle='--')
plt.title("Metrics across Different Thresholds")
plt.xlabel("Probability Threshold")
plt.ylabel("Score")
plt.legend()
plt.grid(True)
plt.show()\n

By simply shifting the threshold to a value that maximizes your target metric (e.g., F1 at ~0.3 or 0.4), you achieve better balance without retraining the model.\n

# 21. Ensemble Models and Imbalance

### Concept
Some ensemble models (like Random Forests) natively handle class imbalance very well if properly parameterized.
Using `class_weight="balanced"` in a Random Forest adapts the penalty of trees based on the distribution.\n

In [ ]:
rf_balanced = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_balanced.fit(X_train, y_train)

y_pred_rf = rf_balanced.predict(X_test)
print(f"Random Forest (Balanced) Recall: {recall_score(y_test, y_pred_rf):.4f}")\n

# 22. Complete End-to-End Imbalanced Classification Project

Let's do a complete, realistic workflow.

1. Dataset Creation (Fraud Detection style)
2. Stratified Split
3. Train Models: Baseline, Weighted, Random Oversampling, SMOTE
4. Evaluation and Comparison
5. Threshold Selection\n

In [ ]:
# 1. Dataset 
X_proj, y_proj = make_classification(n_samples=10000, n_features=10, n_informative=4, 
                                     n_redundant=1, weights=[0.98, 0.02], random_state=42)

# 2. Split
X_tr, X_te, y_tr, y_te = train_test_split(X_proj, y_proj, test_size=0.25, stratify=y_proj, random_state=42)

# 3. Models
# a. Baseline
lr_b = LogisticRegression(random_state=42)
lr_b.fit(X_tr, y_tr)

# b. Weighted
lr_w = LogisticRegression(class_weight='balanced', random_state=42)
lr_w.fit(X_tr, y_tr)

# c. ROS
ros2 = RandomOverSampler(random_state=42)
X_tr_ros, y_tr_ros = ros2.fit_resample(X_tr, y_tr)
lr_ros = LogisticRegression(random_state=42)
lr_ros.fit(X_tr_ros, y_tr_ros)

# d. SMOTE
smote2 = SMOTE(random_state=42)
X_tr_sm, y_tr_sm = smote2.fit_resample(X_tr, y_tr)
lr_sm = LogisticRegression(random_state=42)
lr_sm.fit(X_tr_sm, y_tr_sm)

# 4. Evaluate Models (Probabilities for PR-AUC)
models_to_eval = {
    'Baseline': lr_b,
    'Class Weight': lr_w,
    'ROS': lr_ros,
    'SMOTE': lr_sm
}

proj_results = []
pred_probs = {}

for m_name, model in models_to_eval.items():
    preds = model.predict(X_te)
    probs = model.predict_proba(X_te)[:, 1]
    pred_probs[m_name] = probs
    
    proj_results.append({
        "Model": m_name,
        "Precision": precision_score(y_te, preds, zero_division=0),
        "Recall": recall_score(y_te, preds),
        "F1": f1_score(y_te, preds),
        "ROC-AUC": roc_auc_score(y_te, probs),
        "Average Precision": average_precision_score(y_te, probs)
    })

proj_df = pd.DataFrame(proj_results)

# 5. Plotting PR Curves
plt.figure(figsize=(8,6))
for m_name, probs in pred_probs.items():
    p, r, _ = precision_recall_curve(y_te, probs)
    ap = average_precision_score(y_te, probs)
    plt.plot(r, p, label=f"{m_name} (AP={ap:.2f})")

plt.title("Precision-Recall Curve Comparison")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid(True)
plt.show()\n

# 23. Method Comparison

### Results Summary\n

In [ ]:
from IPython.display import display
display(proj_df)\n

In many cases (often in real datasets), SMOTE or Weighting drastically improves Recall, but hurts Precision compared to a Baseline that only barely guesses the positive class occasionally. Average Precision often holds steady or improves slightly, showing that ranking performance of minority classes is enhanced.

**Model Selection:**
* If detecting every fraud case is critical, `Class Weight` or `SMOTE` usually perform best.
* If minimizing false alarms is critical, manipulating the baseline model's `Classification Threshold` upward is often best.\n

# 24. Common Mistakes

> **1. Using accuracy alone:** Hides failure on the minority class. Use Precision, Recall, F1, PR-AUC.
> **2. Not stratifying train/test split:** Breaks natural distributions. Always use `stratify=y`.
> **3. Applying SMOTE before splitting:** Causes massive Data Leakage. Split first, SMOTE train only.
> **4. Resampling the test set:** The test set must perfectly mirror reality. Do not artificially alter it.
> **5. Data leakage during cross-validation:** Applying SMOTE to full K-fold data beforehand. Use `imblearn.pipeline`.
> **6. Oversampling validation/test data:** False metric inflation.
> **7. Ignoring Precision-Recall:** Looking only at ROC-AUC which can be artificially high.
> **8. Using an unsuitable threshold:** Blindly trusting `0.5`. Adjust it based on business needs.
> **9. Assuming balanced data is always better:** Sometimes models perform perfectly well on natural imbalance.
> **10. Overfitting with oversampling:** Standard Random Oversampling creates duplicate records. Use SMOTE or Class Weights.
> **11. Throwing away too much data with undersampling:** Losing major patterns from Class 0. Use carefully or ensemble.
> **12. Choosing a model based only on one metric:** Optimization should align carefully with domain impacts (cost of FP vs FN).\n

# 25. Interview Questions

1. **What is class imbalance?** Unequal representation of target classes (e.g. 99% vs 1%).
2. **Why is accuracy misleading?** A 99% accurate model might miss 100% of the minority class (e.g. predicting everything majority).
3. **What is the majority class?** The class with the highest frequency.
4. **What is the minority class?** The class with the lowest frequency (often the one we care about).
5. **Precision vs Recall?** Precision = quality of positive predictions. Recall = quantity of actual positives found.
6. **What is F1?** Harmonic mean of Precision and Recall.
7. **What is class weighting?** Penalizing algorithm heavily for misclassifying the minority class.
8. **What is undersampling?** Removing random majority samples to balance sizes.
9. **What is oversampling?** Duplicating minority samples to balance sizes.
10. **What is SMOTE?** Creating completely synthetic minority samples using Nearest Neighbors.
11. **Why should SMOTE only be applied to training data?** To prevent data leakage and ensure testing is valid.
12. **What is data leakage?** When information from outside the training dataset enters the model.
13. **ROC-AUC vs PR-AUC?** ROC evaluates TPR vs FPR. PR-AUC evaluates Precision vs Recall (better for minority classes).
14. **Why is PR-AUC useful for imbalanced data?** It doesn't factor in True Negatives (majority class), focusing solely on minority detection.
15. **What is threshold tuning?** Shifting decision boundary (e.g., >0.3 vs >0.5) to favor precision or recall.
16. **How does `class_weight="balanced"` work?** Uses inverse proportions of class frequencies as loss penalties.
17. **Advantages/disadvantages of SMOTE?** Pro: No information loss, lower overfitting than ROS. Con: Synthetics might lie in majority regions, blurring decision boundaries.
18. **How do you evaluate an imbalanced classifier?** Look at Confusion Matrix, Precision, Recall, F1, PR-AUC, business cost of errors.
19. (Continued...) (19-30 generally revolve around deeper implications of above: e.g. Impact of noisy datasets on SMOTE, using embeddings, tree-methods robustness, anomaly detection proxies).\n

# 26. Quick Revision Cheat Sheet

### Important Concepts
| Concept | Definition |
|---------|---------|
| **Class Imbalance** | Classes not equally distributed |
| **Stratification** | Forcing train/test splits to mimic original proportions |
| **Class Weight** | Math penalty scaling during training loss |
| **Undersampling** | Deleting majority records |
| **Oversampling** | Copying minority records |
| **SMOTE** | Faking new minority records smoothly |
| **Threshold Tuning** | Shifting from 0.5 to target specific Precision or Recall |

### Correct Workflow
1. Dataset loaded.
2. `train_test_split(..., stratify=y)`
3. Set aside Test Data.
4. Use Pipeline for Cross Validation on Training Data.
5. In Pipeline: Resample or Weight (e.g., SMOTE) -> Model Training.
6. Check metrics across CV folds.
7. Tune threshold if necessary.
8. Final single evaluation on untouched Test Data.\n

# 27. Practice Problems

1. Create a `make_classification` dataset with 1% minority presence.
2. Calculate and print the exact class distribution using Pandas.
3. Train a generic SVM model and show why accuracy is highly misleading using the confusion matrix.
4. Perform a `RandomSearchCV` using Stratified K-fold CV.
5. Train a baseline Logistic Regression. Report Precision and Recall.
6. Instead of resampling, apply `class_weight` and report the metrics shift.
7. Apply Random Oversampling and evaluate on an un-sampled test set.
8. Apply Random Undersampling and compare to Oversampling.
9. Construct an `imblearn` Pipeline using SMOTE and RandomForest.
10. Extract probabilities and plot a beautiful Precision-Recall curve.
11. Compute metrics for thresholds: `[0.15, 0.35, 0.55, 0.75]`.
12. Write a short paragraph comparing all tested balancing techniques for your data.\n

### Key Takeaways
Always use **stratification**. Never evaluate with **accuracy alone** if classes are imbalanced. Never **resample** the testing set. Know the business cost of a **False Positive** vs **False Negative** and choose metrics accordingly!

---

## Next Notebook
`15_feature_engineering.ipynb`

*Our next notebook will focus on Feature Engineering: creating, transforming, selecting, and improving features for Machine Learning models.*\n